# Industry-Specific Large Language Model (LLM) Bot

## Capstone Project

### Industry:
Technology and Information Technology (IT)

### Step 1:- Objective
The goal of this project is to develop an Industry-Specific Large Language Model (LLM) Bot capable of answering questions related to the Technology and Information Technology (IT) domain. Instead of training a language model from scratch, a pre-trained open-source Large Language Model from Hugging Face will be fine-tuned using domain-specific data.

The chatbot will be able to understand user queries related to programming, databases, networking, cloud computing, cybersecurity, operating systems, and software development, and generate accurate, context-aware responses.

This notebook demonstrates the complete workflow including:

- Environment Setup
- Dataset Preparation
- Data Preprocessing
- Model Selection
- Fine-Tuning using LoRA
- Model Evaluation
- Chatbot Inference

# Step 2 - Installing Required Libraries

This project uses the Hugging Face ecosystem for loading and fine-tuning a pre-trained Large Language Model.

The following libraries are required:

- transformers - Load and use pre-trained models
- datasets - Handle training datasets
- peft - Parameter Efficient Fine-Tuning (LoRA)
- trl - Supervised Fine-Tuning
- accelerate - Efficient training
- bitsandbytes - 4-bit quantization
- sentencepiece - Tokenization support
- gradio - Chatbot interface

These libraries will be installed in the next code cell.

In [1]:
!pip -q install -U transformers
!pip -q install -U datasets
!pip -q install -U peft
!pip -q install -U trl
!pip -q install -U accelerate
!pip -q install -U bitsandbytes
!pip -q install -U sentencepiece
!pip -q install -U gradio

# Step 3 - Importing Required Libraries

After installing the required packages, we import all the necessary Python modules.

These libraries will be used for:

- Loading the dataset
- Loading the pre-trained model
- Fine-tuning using LoRA
- Tokenization
- Training
- Building the chatbot

In [2]:
import torch
import transformers
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model
)

print("All libraries imported successfully!")

All libraries imported successfully!


# Step 4 - Selecting the Pre-trained Model

For this project we will use an open-source Large Language Model from Hugging Face.

Model Selected:

**TinyLlama-1.1B-Chat-v1.0**

Reasons for selection:

- Lightweight
- Suitable for Google Colab T4 GPU
- Open-source
- Good conversational performance
- Supports LoRA fine-tuning

In [3]:
from huggingface_hub import login

login()

In [4]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '667e65c4b3072cb50c909f2b', 'name': 'Yashman', 'fullname': 'YAS', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1785542400, 'isPro': False, 'avatarUrl': '/avatars/59f765f85ca9eeccbfa057603fc4d468.svg', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-08-02T10:43:01.000Z'}}


# Step 6 - Loading the Pre-trained Large Language Model

In this project, we use **Qwen2.5-1.5B-Instruct**, an open-source instruction-tuned Large Language Model from Hugging Face.

Reasons for choosing this model:

- Lightweight enough for Google Colab T4 GPU
- Excellent instruction-following capability
- Strong conversational performance
- Compatible with LoRA and QLoRA fine-tuning
- Open-source and widely used

To reduce GPU memory usage, the model is loaded using **4-bit quantization (BitsAndBytes)**.

In [5]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


# Step 7 - Verifying the Model

Before fine-tuning, we verify that the model and tokenizer have been loaded successfully.

The following cell prints basic information about the model, including its architecture and vocabulary size.

In [6]:
print("Model Name:", MODEL_NAME)

print("\nVocabulary Size:", tokenizer.vocab_size)

print("\nModel Architecture:")
print(model.config.architectures)

print("\nModel is ready for fine-tuning!")

Model Name: Qwen/Qwen2.5-1.5B-Instruct

Vocabulary Size: 151643

Model Architecture:
['Qwen2ForCausalLM']

Model is ready for fine-tuning!


In [7]:
import torch
print(torch.cuda.get_device_name(0))

Tesla T4


# Step 8 - Preparing the Training Dataset

The quality of an LLM depends heavily on the quality of its training data.

For this project, we prepare an instruction-response dataset focused on the Information Technology (IT) domain.

Each training example contains:

- Instruction (User Question)
- Response (Expected Answer)

This format enables the model to learn how to answer technical questions in a conversational manner.

The dataset will later be converted into a Hugging Face Dataset object for fine-tuning.

In [8]:
training_data = [
    {
        "instruction": "What is Python?",
        "response": "Python is a high-level programming language widely used for web development, automation, artificial intelligence, machine learning, and data analysis."
    },
    {
        "instruction": "What is SQL?",
        "response": "SQL stands for Structured Query Language. It is used to create, manage, and query relational databases."
    },
    {
        "instruction": "What is Machine Learning?",
        "response": "Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from data without being explicitly programmed."
    },
    {
        "instruction": "What is Artificial Intelligence?",
        "response": "Artificial Intelligence is the simulation of human intelligence by machines capable of learning, reasoning, and decision-making."
    }
]

print("Total Examples:", len(training_data))

Total Examples: 4


# Step 9 - Loading the Industry Dataset

A high-quality dataset is essential for training an effective Large Language Model.

Instead of manually writing a few examples, we use a publicly available instruction dataset from Hugging Face.

The dataset will later be filtered and formatted for Information Technology (IT) related conversations.

Using an existing dataset provides better diversity and improves the chatbot's ability to answer a wide range of technical questions.

In [9]:
from datasets import load_dataset

In [10]:
dataset = load_dataset(
    "yahma/alpaca-cleaned",
    split="train"
)

print(dataset)

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 51760
})


# Step 10 - Filtering Technology Related Data

The Alpaca dataset contains instructions from many domains.

Since this project focuses on the Technology and Information Technology industry, we filter the dataset to retain only technology-related examples.

This improves the relevance of the training data and aligns the chatbot with the project objective.

In [11]:
keywords = [
    "python",
    "sql",
    "database",
    "linux",
    "network",
    "programming",
    "computer",
    "software",
    "coding",
    "algorithm",
    "artificial intelligence",
    "machine learning",
    "data",
    "cloud",
    "cyber",
    "api",
    "web",
    "javascript",
    "java",
    "c++",
    "git",
    "github"
]

def is_it_related(example):
    text = (
        example["instruction"] + " " + example["input"]
    ).lower()

    return any(keyword in text for keyword in keywords)

it_dataset = dataset.filter(is_it_related)

Filter:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [12]:
print("Total IT Examples:", len(it_dataset))

Total IT Examples: 6047


I started with the Alpaca instruction dataset and filtered it to retain only IT-related examples using keyword-based filtering. This created a domain-specific dataset suitable for fine-tuning.

# Step 11 - Exploring the Dataset

Before fine-tuning the model, it is important to inspect the dataset.

This step helps verify that the data has been loaded correctly and allows us to understand the structure of the instruction-response pairs.

Exploring the dataset also helps identify any inconsistencies before training begins.

In [13]:
print(it_dataset[0])

{'output': 'The Commodore 64 was a highly successful 8-bit home computer manufactured by Commodore Business Machine (CBM) in 1982, with sales amounting to approximately 17 million units sold between 1983-1986. It dominated the market with between 30% and 40% share and outsold its competitors, including IBM PC clones, Apple Computers, and Atari computers. At its peak, CBM was building 400,000 C64s a month for a couple of years.', 'input': '', 'instruction': 'Write a concise summary of the following:\n"Commodore 64 (commonly known as the C64 or CBM 64) was manufactured by Commodore Business Machine (CBM) in August 1982 with a starting price of $595. It was an 8-bit home computer with remarkable market success. Between 1983-1986, C64 sales amounted to about 17 million units sold, making them the best-selling single personal computer model of all time in 1983-1986. \n\nAdditionally, the Commodore 64 dominated the market with between 30% and 40% share and 2 million units sold per year, outs

In [14]:
print(it_dataset[1])

{'output': 'The capital city of France is Paris.', 'input': '', 'instruction': 'What is the capital of France?'}


In [15]:
print(it_dataset[2])

{'output': "The motherboard, also known as the mainboard or system board, is the central printed circuit board in a computer. It serves as the backbone or foundation for a computer, connecting all the different components such as the CPU, RAM, storage drives, expansion cards, and peripherals. The motherboard manages communication and data transfer between these components, allowing them to work together and perform their designated tasks.\n\nThe motherboard also includes important circuitry such as the power regulation circuit that provides power to the different components, and the clock generator which synchronizes the operation of these components. It also contains the BIOS (basic input/output system), which is a firmware that controls the boot process and provides an interface for configuring and managing the computer's hardware. Other features on a motherboard may include built-in networking, audio, and video capabilities.\n\nOverall, the function of a computer motherboard is to p

# Step 12 - Formatting the Dataset

The language model expects each training example to follow a conversational format.

Each example is converted into a prompt consisting of:

- Instruction
- Response

This formatting enables the model to learn how to generate accurate responses for user queries.

In [16]:
def formatting(example):

    text = f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}
"""

    return {"text": text}

In [17]:
formatted_dataset = it_dataset.map(formatting)

Map:   0%|          | 0/6047 [00:00<?, ? examples/s]

In [18]:
print(formatted_dataset[0]["text"])

### Instruction:
Write a concise summary of the following:
"Commodore 64 (commonly known as the C64 or CBM 64) was manufactured by Commodore Business Machine (CBM) in August 1982 with a starting price of $595. It was an 8-bit home computer with remarkable market success. Between 1983-1986, C64 sales amounted to about 17 million units sold, making them the best-selling single personal computer model of all time in 1983-1986. 

Additionally, the Commodore 64 dominated the market with between 30% and 40% share and 2 million units sold per year, outselling the IBM PC clones, Apple Computers, and Atari computers. Adding to their success, Sam Tramiel (former Atari president), during an interview in 1989, said they were building 400,000 C64s a month for a couple of years. "

### Response:
The Commodore 64 was a highly successful 8-bit home computer manufactured by Commodore Business Machine (CBM) in 1982, with sales amounting to approximately 17 million units sold between 1983-1986. It domina

# Step 13 - Preparing the Final Training Dataset

The dataset is simplified by retaining only the formatted text column.

This reduces unnecessary information and prepares the dataset for efficient fine-tuning.

In [19]:
formatted_dataset = formatted_dataset.remove_columns(
    ["instruction", "input", "output"]
)

In [20]:
formatted_dataset

Dataset({
    features: ['text'],
    num_rows: 6047
})

# Step 14 - Selecting Conversational IT Examples

Although the filtered dataset contains technology-related examples, many entries are summarization or writing tasks.

Since the objective of this project is to build an IT conversational assistant, the dataset is further refined to retain instruction-response examples that resemble real user questions.

This improves the chatbot's ability to answer technical queries naturally.

In [21]:
question_words = [
    "what",
    "how",
    "why",
    "when",
    "where",
    "which",
    "who",
    "define",
    "describe",
    "explain",
    "difference"
]

def conversational(example):
    instruction = example["text"].lower()

    return any(
        instruction.startswith(f"### instruction:\n{word}")
        for word in question_words
    )

chat_dataset = formatted_dataset.filter(conversational)

print("Conversational Examples:", len(chat_dataset))

Filter:   0%|          | 0/6047 [00:00<?, ? examples/s]

Conversational Examples: 1312


# Step 15 - Loading the Pre-trained Language Model

In this project, the Qwen2.5-0.5B-Instruct model is selected as the base Large Language Model.

Reasons for choosing this model:

- Lightweight and suitable for Google Colab T4 GPU
- Open-source and freely available
- Strong instruction-following capability
- Compatible with Parameter-Efficient Fine-Tuning (PEFT)
- Supports QLoRA, reducing GPU memory requirements while maintaining good performance

The model is loaded using 4-bit quantization to minimize memory usage during training.

In [22]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [23]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [24]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [25]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm(

# Step 16 - Configuring LoRA

Instead of updating every parameter in the model, LoRA (Low-Rank Adaptation) updates only a small number of trainable parameters.

This approach:

- Reduces GPU memory usage
- Speeds up training
- Preserves the original model weights
- Makes fine-tuning practical on consumer hardware

In [26]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

model = prepare_model_for_kbit_training(model)

In [27]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [28]:
model = get_peft_model(model, lora_config)

In [29]:
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


# Step 17 - Verifying the Model

Before fine-tuning, the model configuration is verified to ensure that the tokenizer and LoRA adapters have been successfully loaded.

This step also confirms the vocabulary size and model architecture.

In [30]:
print("Model Name:", MODEL_NAME)

print("\nVocabulary Size:", tokenizer.vocab_size)

print("\nModel Architecture:")
print(model.config.architectures)

print("\nEOS Token:", tokenizer.eos_token)
print("PAD Token:", tokenizer.pad_token)

Model Name: TinyLlama/TinyLlama-1.1B-Chat-v1.0

Vocabulary Size: 32000

Model Architecture:
['LlamaForCausalLM']

EOS Token: </s>
PAD Token: </s>


In [31]:
tokenizer.pad_token = tokenizer.eos_token

# Step 18 - Tokenizing the Dataset

Large Language Models cannot process raw text directly. Therefore, the conversational dataset must be converted into numerical token IDs using the tokenizer associated with the selected model.

Each instruction-response pair is tokenized with:

- Maximum sequence length of 512 tokens
- Automatic truncation for longer sequences
- Padding to ensure uniform input length

The resulting tokenized dataset will be used during the fine-tuning process.

In [32]:
MAX_LENGTH = 512

In [33]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

In [34]:
tokenized_dataset = chat_dataset.map(
    tokenize_function,
    batched=True
)


Map:   0%|          | 0/1312 [00:00<?, ? examples/s]

In [35]:
print(tokenized_dataset)

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 1312
})


In [36]:
print(tokenized_dataset[0])

{'text': '### Instruction:\nWhat is the capital of France?\n\n### Response:\nThe capital city of France is Paris.\n', 'input_ids': [1, 835, 2799, 4080, 29901, 13, 5618, 338, 278, 7483, 310, 3444, 29973, 13, 13, 2277, 29937, 13291, 29901, 13, 1576, 7483, 4272, 310, 3444, 338, 3681, 29889, 13, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,

In [37]:
import transformers
import trl
import peft

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)

Transformers: 5.12.1
TRL: 1.7.0
PEFT: 0.19.1


In [38]:
import sys

print("Python Version:", sys.version)

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [39]:
!pip show transformers trl peft accelerate bitsandbytes datasets

Name: transformers
Version: 5.12.1
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: peft, sentence-transformers, trl
---
Name: trl
Version: 1.7.0
Summary: Train transformer language models with reinforcement learning.
Home-page: https://github.com/huggingface/trl
Author: 
Author-email: Leandro von Werra <leandro.vonwerra@gmail.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: accelerate, datasets, jinja2,

In [40]:
import transformers
import trl
import peft

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)

Transformers: 5.12.1
TRL: 1.7.0
PEFT: 0.19.1


# Step 19 - Preparing the Tokenizer

Before fine-tuning, the tokenizer is configured for causal language modeling.

Since the selected model does not define a dedicated padding token, the end-of-sequence (EOS) token is used as the padding token. This ensures that all input sequences have a uniform length without introducing invalid tokens during training.

The tokenizer is also configured to pad sequences on the right side, which is recommended for causal language models.

In [41]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Pad Token:", tokenizer.pad_token)
print("Padding Side:", tokenizer.padding_side)

Pad Token: </s>
Padding Side: right


# Step 20 - Creating the Data Collator

The data collator dynamically prepares batches during training by applying the tokenizer and generating tensors.

For causal language modeling, the collator ensures that the model learns to predict the next token in the response while efficiently handling variable-length sequences.

In [42]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator created successfully.")

Data collator created successfully.


In [43]:
from trl import SFTTrainer
import inspect

print(inspect.signature(SFTTrainer))

(model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callback.TrainerCallback] | None = None, optimizers: tuple[torch.optim.optimizer.Optimizer | None, torch.optim.lr_

In [44]:
help(SFTTrainer)

Help on class SFTTrainer in module trl.trainer.sft_trainer:

class SFTTrainer(trl.trainer.base_trainer._BaseTrainer)
 |  SFTTrainer(model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[tran

Step 19 – Training Configuration

This step defines the training hyperparameters used during fine-tuning.

Important settings include:

• 3 training epochs
• Learning rate of 2e-4
• Batch size of 2
• Gradient accumulation to reduce GPU memory usage
• Logging every 10 steps
• Saving the final model after training

These settings are suitable for fine-tuning a 0.5B parameter model on a Tesla T4 GPU using QLoRA.

In [65]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./IT_LLM",

    num_train_epochs=3,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=2,

    learning_rate=2e-4,

    logging_steps=10,

    save_strategy="epoch",

    bf16=True, # Set bf16 to True
    fp16=False, # Set fp16 to False

    report_to="none"
)

print(training_args)

SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_ch

Step 20 – Creating the Supervised Fine-Tuning (SFT) Trainer

The SFT Trainer manages the complete fine-tuning process.

It performs the following tasks:

• Tokenizes the dataset
• Creates training batches
• Feeds the data to the model
• Computes training loss
• Updates only the LoRA parameters
• Saves checkpoints during training

The formatted conversational dataset is used directly for supervised fine-tuning.

In [66]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
)

In [47]:
print(trainer)

Step 21 – Fine-Tuning the Model

The trainer now begins supervised fine-tuning using the formatted IT conversational dataset.

During training, only the LoRA adapter weights are updated while the original Qwen model remains frozen.

This significantly reduces GPU memory usage while producing an industry-specific conversational model.

In [48]:
model.config.use_cache = False

trainer.accelerator.scaler = None

In [49]:
!pip uninstall -y transformers trl accelerate peft bitsandbytes

!pip install -q \
transformers==4.51.3 \
trl==0.17.0 \
peft==0.15.2 \
accelerate==1.5.2 \
bitsandbytes==0.45.5

Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: trl 1.7.0
Uninstalling trl-1.7.0:
  Successfully uninstalled trl-1.7.0
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.6 MB/s eta 0:00:00
   ━━━━━━

In [50]:
import transformers
import trl
import peft
import accelerate
import bitsandbytes

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

Transformers: 5.12.1
TRL: 1.7.0
PEFT: 0.19.1
Accelerate: 1.14.0
BitsAndBytes: 0.49.2


In [51]:
!pip uninstall -y transformers trl accelerate peft bitsandbytes

!pip install -q \
transformers==4.51.3 \
trl==0.17.0 \
peft==0.15.2 \
accelerate==1.5.2 \
bitsandbytes==0.45.5

Found existing installation: transformers 4.51.3
Uninstalling transformers-4.51.3:
  Successfully uninstalled transformers-4.51.3
Found existing installation: trl 0.17.0
Uninstalling trl-0.17.0:
  Successfully uninstalled trl-0.17.0
Found existing installation: accelerate 1.5.2
Uninstalling accelerate-1.5.2:
  Successfully uninstalled accelerate-1.5.2
Found existing installation: peft 0.15.2
Uninstalling peft-0.15.2:
  Successfully uninstalled peft-0.15.2
Found existing installation: bitsandbytes 0.45.5
Uninstalling bitsandbytes-0.45.5:
  Successfully uninstalled bitsandbytes-0.45.5


In [52]:
import transformers
import trl
import peft
import accelerate
import bitsandbytes

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

Transformers: 5.12.1
TRL: 1.7.0
PEFT: 0.19.1
Accelerate: 1.14.0
BitsAndBytes: 0.49.2


In [53]:
import sys
print(sys.executable)

/usr/bin/python3


In [54]:
!pip show transformers
!pip show trl
!pip show peft
!pip show accelerate
!pip show bitsandbytes

Name: transformers
Version: 4.51.3
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers, trl
Name: trl
Version: 0.17.0
Summary: Train transformer language models with reinforcement learning.
Home-page: https://github.com/huggingface/trl
Author: Leandro von Werra
Author-email: leandro.vonwerra@gmail.com
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: accelerate, datasets, rich, transformers
Required-by: 
Name: peft
Version: 0.15.2
Summary: Parameter-Efficient Fine-

In [55]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
12.8
True
Tesla T4


In [56]:
import transformers
import trl
import os

print("Transformers location:")
print(transformers.__file__)

print("\nTRL location:")
print(trl.__file__)

print("\nPython executable:")
import sys
print(sys.executable)

print("\nInstalled packages:")
!pip show transformers trl peft accelerate bitsandbytes

Transformers location:
/usr/local/lib/python3.12/dist-packages/transformers/__init__.py

TRL location:
/usr/local/lib/python3.12/dist-packages/trl/__init__.py

Python executable:
/usr/bin/python3

Installed packages:
Name: transformers
Version: 4.51.3
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers, trl
---
Name: trl
Version: 0.17.0
Summary: Train transformer language models with reinforcement learning.
Home-page: https://github.com/huggingface/trl
Author: Leandro von Werra
Author-email: le

In [57]:
!which python
!which pip
!python -m pip show transformers

/usr/local/bin/python
/usr/local/bin/pip
Name: transformers
Version: 4.51.3
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers, trl


In [58]:
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=training_args,
    processing_class=tokenizer,
)

In [59]:
import torch

dtypes = {}

for name, param in model.named_parameters():
    dt = str(param.dtype)
    dtypes[dt] = dtypes.get(dt, 0) + 1

print(dtypes)

{'torch.float32': 47, 'torch.uint8': 154, 'torch.bfloat16': 176}


In [60]:
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)

print("bf16 supported:", torch.cuda.is_bf16_supported())

print("Default dtype:", torch.get_default_dtype())

Torch: 2.11.0+cu128
CUDA: 12.8
bf16 supported: True
Default dtype: torch.float32


In [61]:
print(training_args.fp16)
print(training_args.bf16)

True
False


In [62]:
print(model.config.torch_dtype)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


torch.bfloat16


In [67]:
trainer.train()

Step,Training Loss
10,1.291198
20,1.231608
30,1.166470
40,1.178123
50,1.248109
60,1.136851
70,1.129588
80,1.128456
90,1.079623
100,1.213776


TrainOutput(global_step=4536, training_loss=1.0555113622330694, metrics={'train_runtime': 5542.5282, 'train_samples_per_second': 3.273, 'train_steps_per_second': 0.818, 'total_flos': 3.745475094922445e+16, 'train_loss': 1.0555113622330694, 'entropy': 1.0628705124060314, 'num_tokens': 4429125.0, 'mean_token_accuracy': 0.7172608822584152, 'epoch': 3.0})

Step 22 – Saving the Fine-Tuned Model

After completing the fine-tuning process, the trained LoRA adapters and tokenizer are saved locally.

Saving the model allows it to be reloaded later without repeating the training process. This makes deployment, inference, and future improvements more efficient.

The saved model will be used to build the Industry-Specific IT Chatbot in the following steps.

In [68]:
# Save the fine-tuned model

trainer.save_model("./IT_LLM_Model")
tokenizer.save_pretrained("./IT_LLM_Model")

print("✅ Fine-tuned model saved successfully.")

✅ Fine-tuned model saved successfully.


Step 23 – Building the Industry-Specific IT Chatbot

The fine-tuned Large Language Model is now used to build an interactive chatbot capable of answering Information Technology related questions.

A helper function is created that accepts a user's question, formats it into the instruction-response prompt used during training, generates a response using the fine-tuned model, and returns the generated answer.

This function serves as the core inference engine of the chatbot.

In [69]:
def IT_Chatbot(question):

    prompt = f"""### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

Step 24 – Interactive Chatbot Interface

To make the model easier to use, an interactive chatbot interface is created using Gradio.

Gradio provides a simple web-based interface where users can enter Information Technology related questions and receive responses from the fine-tuned Large Language Model in real time.

This demonstrates the practical deployment of the chatbot and completes the development phase of the project.

In [72]:
import gradio as gr

def chatbot(user_input):
    return IT_Chatbot(user_input)

interface = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask any IT related question..."
    ),
    outputs="text",
    title="Industry-Specific IT LLM Bot",
    description="Fine-Tuned Large Language Model for Information Technology"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d44a16af1b18fad085.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Step 25 – Model Evaluation

The chatbot was evaluated using multiple Information Technology related questions covering programming, databases, cloud computing, cybersecurity, networking, software development, and artificial intelligence.

The responses generated by the model demonstrated that it could understand technical queries and produce coherent, context-aware answers. Fine-tuning using LoRA enabled the model to specialize in Information Technology while requiring only a small number of trainable parameters, making the training process efficient on limited hardware.

Although the chatbot performs well for general IT concepts, its performance can be further improved by training on larger and more diverse industry-specific datasets, increasing the number of high-quality conversational examples, and incorporating more advanced evaluation techniques.

Conclusion

In this capstone project, an Industry-Specific Large Language Model (LLM) Bot was successfully developed for the Information Technology (IT) domain using a pre-trained open-source language model from Hugging Face.

The project involved collecting and filtering an IT-specific dataset, preprocessing the data into an instruction-response format, applying Parameter-Efficient Fine-Tuning (LoRA), and training the model using supervised fine-tuning on Google Colab with GPU acceleration.

After fine-tuning, the model was used to build an interactive chatbot capable of answering Information Technology related questions in a conversational manner. The chatbot demonstrated its ability to understand technical concepts and generate relevant responses across topics such as programming, databases, networking, cloud computing, cybersecurity, operating systems, and artificial intelligence.

This project illustrates the practical application of Large Language Models in creating domain-specific conversational assistants while highlighting the effectiveness of parameter-efficient fine-tuning techniques for developing intelligent AI systems using limited computational resources.